# Projet d'Actuariat IMAFA 2026-2027
## Logiciel d'Assurance de Risque Météorologique
**Polytech Nice Sophia (MAM - IMAFA)**  
**Enseignant :** Marius AKRE  
**Auteurs :** Amiel Metier, Tahitoa Senis, Nicolas Muratore  

---

### Objectifs de la démonstration :
1. **Modélisation Actuarielle** : Formaliser l'impact économique de la pluie journalière ($CA_t^{pl}$, $R_t^{pl}$, seuil $pl^*$).
2. **Département Tarification (Pricing)** : Calculer la prime pure $\mathbb{E}[L]$, l'écart-type des pertes et la prime commerciale sur l'historique météo réel.
3. **Département Commercial (Devis)** : Émettre la proposition tarifaire et le devis contractuel.
4. **Département R&D (Analyse Rétrospective)** : Valider l'assurance sur l'exemple du sujet (Nice 1998) avec visualisations interactives.

In [ ]:
import os
import sys
import numpy as np
import pandas as pd

# Ajout de la racine du projet au PYTHONPATH
sys.path.insert(0, os.path.abspath(".."))

from src.simulation import ClientProfile, compute_daily_impact, simulate_year
from src.pricing import price_contract_historical
from src.weather_api import get_coordinates, fetch_historical_precipitation
from src.backtesting import run_retrospective_analysis
from src.visualisations import plot_retrospective_comparison, plot_cumulative_results
from src.pdf_generator import generate_devis_pdf

## 1. Définition du Profil Client (Section I)

On définit le commerçant type du sujet (ex: restaurant/café à Nice) :
- $CA = 1\,000$ € (Chiffre d'affaires journalier maximum par temps sec)
- $Cf = 400$ € (Coûts fixes journaliers incompressibles)
- $\overline{pl} = 10$ mm (Niveau de pluie pivot entraînant l'arrêt complet de l'activité, $CA = 0$)

Le seuil de rentabilité journalier est donné par :
$$pl^* = \overline{pl} \left(1 - \frac{Cf}{CA}\right)$$

In [ ]:
profile = ClientProfile(
    ca_max=1000.0,
    cf=400.0,
    pl_pivot=10.0,
    city="Nice",
    subscription_date="1998-01-01"
)

seuil_perte = profile.pl_pivot * (1.0 - profile.cf / profile.ca_max)
print(f"Client : {profile.city}")
print(f"Seuil de perte journalière (pl*) : {seuil_perte:.2f} mm/jour")
print(f"-> Au-delà de {seuil_perte:.2f} mm de pluie, le résultat d'exploitation journalier devient négatif et l'assurance indemnise.")

## 2. Données Météorologiques Réelles (Nice)

Interrogation des coordonnées géographiques et récupération des précipitations journalières réelles pour Nice en 1998.

In [ ]:
coords = get_coordinates(profile.city)
lat, lon, station = coords
print(f"Station identifiée : {station} (lat={lat:.3f}, lon={lon:.3f})")

df_1998 = fetch_historical_precipitation(lat, lon, "1998-01-01", "1998-12-31", cache_dir="../data")
print(f"Nombre de jours relevés en 1998 : {len(df_1998)}")
print(f"Cumul pluviométrique annuel     : {df_1998['precipitation'].sum():.1f} mm")
print(f"Précipitation journalière max   : {df_1998['precipitation'].max():.1f} mm")

## 3. Simulation Journalière et Règle d'Indemnisation

L'assurance garantit un résultat journalier positif ou nul ($R_t \ge 0$).  
L'indemnité versée chaque jour de sinistre ($pl_t > pl^*$) est :
$$I_t = \max(0, -R_t^{pl})$$

In [ ]:
sim_1998 = simulate_year(df_1998["precipitation"], profile)
sim_1998["date"] = df_1998["date"]

nb_jours_sinistres = (sim_1998["indemnity"] > 0).sum()
pertes_totales_1998 = sim_1998["indemnity"].sum()
res_non_assure = sim_1998["r_nass"].sum()
res_assure_brut = sim_1998["r_ass_brut"].sum()

print(f"Jours indemnisés sur l'année       : {nb_jours_sinistres} jours")
print(f"Résultat annuel non assuré (R_nass): {res_non_assure:,.2f} €")
print(f"Pertes réelles évitées (L)         : {pertes_totales_1998:,.2f} €")
print(f"Résultat annuel assuré brut (R_ass): {res_assure_brut:,.2f} €")

## 4. Tarification Prospective (Département Tarification)

Estimation de la prime sur un historique pluriannuel avec chargement de sécurité et frais de gestion.

In [ ]:
# Simulation sur un dictionnaire d'années
df_hist = fetch_historical_precipitation(lat, lon, "1990-01-01", "1997-12-31", cache_dir="../data")
df_hist["year"] = df_hist["date"].dt.year
hist_dict = {yr: grp["precipitation"] for yr, grp in df_hist.groupby("year")}

pricing_res = price_contract_historical(
    hist_dict,
    profile,
    safety_margin_rate=0.10,
    expense_rate=0.15
)

print(f"Prime pure technique E[I]       : {pricing_res['pure_premium']:,.2f} €")
print(f"Écart-type des sinistres        : {pricing_res['std_indemnity']:,.2f} €")
print(f"Chargement de sécurité          : {pricing_res['safety_loading']:,.2f} €")
print(f"Prime commerciale annuelle      : {pricing_res['commercial_premium']:,.2f} €")

## 5. Analyse Rétrospective — Exemple Nice 1998 (Département R&D)

Vérification de l'exemple officiel du sujet :
- Année : 1998 ($a$)
- Prime payée au début de l'année : $P = 3\,000$ €
- Comparaison $R_{nass}$ vs $R_{ass}$ et $R_{net} = R_{ass} - P$.

In [ ]:
retro = run_retrospective_analysis(df_1998["precipitation"], profile, annual_premium=3000.0)

print("=== BILAN RÉTROSPECTIF 1998 ===")
print(f"Résultat Non Assuré (R_nass)    : {retro['r_nass']:,.2f} €")
print(f"Prime a priori versée (P)       : {retro['premium']:,.2f} €")
print(f"Résultat Garanti Brut (R_ass)   : {retro['r_ass']:,.2f} €")
print(f"Pertes réelles évitées          : {retro['pertes_evitees']:,.2f} €")
print(f"Résultat Net Assuré (R_net)     : {retro['r_net']:,.2f} €")
print(f"Gain net grâce à l'assurance    : {retro['gain_net_client']:+,.2f} €")
print(f"Assurance favorable au client   : {'OUI' if retro['est_favorable'] else 'NON'}")

### Graphique de Comparaison Rétrospective (Plotly)

In [ ]:
fig_bar = plot_retrospective_comparison(retro, 1998)
fig_bar.show()

### Trajectoire Temporelle Cumulée du Résultat Journalier

In [ ]:
retro_details = retro["details_journaliers"]
retro_details["date"] = df_1998["date"]
fig_cumul = plot_cumulative_results(retro_details, 1998)
fig_cumul.show()